# 3.4 假设检验工具包

## 3.4.1 正态性检验

正态性检验的目的是为了检测一组数据是否服从正态分布，是否表现出正态分布的特性。正态性检验的方法有很多，包括QQ图、KS检验、SW检验、JB检验等等。这里当然不可能把它们全部讲出来，但可以对一些常见方法进行简要介绍。

在`scipy.stats`模块中，进行正态性检验的常用方法包括Shapiro-Wilk检验、Anderson-Darling检验、Jarque-Bera检验等。不过，需要注意的是，`scipy.stats`直接提供了Shapiro-Wilk检验（`shapiro`）和Anderson-Darling检验（`anderson`）的实现。

Shapiro-Wilk检验是一种检验样本是否来自正态分布的方法，它对于小样本数据（n < 50）特别有效。

In [ ]:
from scipy.stats import shapiro

# 假设data是你的样本数据
data = np.random.normal(loc=0, scale=1, size=100)  # 生成一个正态分布的随机样本

# 进行Shapiro-Wilk检验
stat, p = shapiro(data)

# 输出结果
print('统计量W:', stat)
print('P值:', p)

# 根据P值判断，如果P值小于显著性水平（如0.05），则拒绝原假设，认为数据不是正态分布的
if p < 0.05:
    print("样本不是正态分布的")
else:
    print("样本数据很可能是正态分布的")

Anderson-Darling检验是另一种检验数据是否服从特定分布（如正态分布）的方法。它适用于较大的样本量。

In [ ]:
from scipy.stats import anderson

# 假设data是你的样本数据
data = np.random.normal(loc=0, scale=1, size=100)  # 生成一个正态分布的随机样本

# 进行Anderson-Darling检验
result = anderson(data, dist='norm')  # 指定检验的正态分布

# 输出结果
print('统计量:', result.statistic)
print('显著性水平对应的临界值:', result.critical_values)
print('显著性水平:', result.significance_level)

# 注意：Anderson-Darling检验不直接给出P值，而是给出统计量和不同显著性水平下的临界值。
# 你需要比较统计量和临界值来判断是否拒绝原假设。

# 例如，如果统计量大于0.05显著性水平下的临界值，则可能拒绝原假设（数据不是正态分布的）
if result.statistic > result.critical_values[2]:  # 这里的2代表5%显著性水平
    print("样本不是正态分布的")
else:
    print("没有足够的证据拒绝样本是正态分布的假设")

## 3.4.2 几种典型的假设检验

在`scipy.stats`模块中，你可以进行多种统计检验，包括卡方独立性检验、独立样本t检验、配对样本t检验、方差分析（ANOVA）等。但是，需要注意的是，`scipy.stats`直接提供了卡方独立性检验、独立样本t检验和配对样本t检验的函数，而方差分析（ANOVA）通常使用`scipy.stats.f_oneway`，而事后多重比较（如Tukey's HSD）则可能需要使用`statsmodels`或其他库。下面我将分别给出这些检验的示例代码。

### 1. 卡方独立性检验

卡方独立性检验用于检验两个分类变量是否独立。卡方独立性检验统计的是离散的相关关系，因为得不得肺癌只有两类离散取值：得或者不得，抽不抽烟也只有两类取值：抽或者不抽。两两组合就有四类人群。统计不同的人群可以列出一个列联表，构造的统计量也是一个服从卡方分布的统计量，因其服从卡方分布所以叫它卡方独立性检验。

In [ ]:
from scipy.stats import chi2_contingency

# 假设有一个2x2的列联表
observed = np.array([[10, 20], [30, 40]])

# 进行卡方独立性检验
chi2, p, dof, expected = chi2_contingency(observed)

print(f'Chi-squared: {chi2}, p-value: {p}')

# 如果p值小于显著性水平（如0.05），则拒绝独立性假设
if p < 0.05:
    print("两个分类变量不独立")
else:
    print("两个分类变量独立")

### 2. 独立样本t检验

独立样本t检验用于比较两个独立样本的均值是否存在显著差异。独立样本t检验适合检验两组不同的样本在某一方面的表现差异。例如，想探究高一学生2000米跑成绩和高三学生2000米跑的成绩差异，这种情况就适合使用独立样本t检验。因为高一学生和高三学生是两批不同的人，它们的男女比例不同、年龄不同、平均身高体重不同……甚至连人数都是不一样的！区别独立样本和配对样本一个最根本的特征就是样本是同一批还是不同的两批，而最直观的特征就是两组样本的数量是否相同。数量不同的两组样本不能构成配对样本。

In [ ]:
from scipy.stats import ttest_ind

# 假设有两个独立样本
sample1 = np.random.normal(loc=0, scale=1, size=100)
sample2 = np.random.normal(loc=0.5, scale=1, size=100)

# 进行独立样本t检验
t_stat, p_val = ttest_ind(sample1, sample2)

print(f'T-statistic: {t_stat}, p-value: {p_val}')

# 如果p值小于显著性水平（如0.05），则拒绝零假设（两个样本均值相等）
if p_val < 0.05:
    print("两个样本的均值存在显著差异")
else:
    print("两个样本的均值没有显著差异")

### 3. 配对样本t检验

配对样本t检验用于比较同一组对象在不同条件下的均值是否存在显著差异。配对样本t检验适合检验同一组样本在进行某一操作前后的状态差异。例如，想探究一笼健康的小白鼠在注射某神经亢奋药物前后的神经活跃性差异，这种情况就适合使用配对样本t检验。因为在注射药物前后，小白鼠始终是同一批小白鼠，没有新的老鼠混进来也没有老鼠逃走，它们只是需要被检测注射药物前后两种不同的状态。

In [ ]:
from scipy.stats import ttest_rel

# 假设有一组对象在两种条件下的测量结果
before = np.random.normal(loc=0, scale=1, size=100)
after = np.random.normal(loc=0.3, scale=1, size=100)

# 进行配对样本t检验
t_stat, p_val = ttest_rel(before, after)

print(f'T-statistic: {t_stat}, p-value: {p_val}')

# 如果p值小于显著性水平（如0.05），则拒绝零假设（两种条件下的均值相等）
if p_val < 0.05:
    print("两种条件下的均值存在显著差异")
else:
    print("两种条件下的均值没有显著差异")

### 4. 方差分析（ANOVA）

方差分析（ANOVA）可以用于两个样本及以上样本之间的比较，并可以用于分离各有关因素并估计其对总变异的作用，以及分析因素间的交互作用。方差分析可以用于均数差别的显著性检验、分离各有关因素并估计其对总变异的作用、分析因素间的交互作用和方差齐性检验等。

方差分析的基本思想是通过比较不同组别之间的平均数差异来确定这些差异是否显著。它利用方差度量每个组别的变异，并将这些变异分解为组内和组间变异。通过比较组间变异和组内变异的比例，可以判断不同组别之间的平均数差异是否具有统计意义。如果组间变异的比例较大，说明组别之间的差异显著。反之，如果组内变异的比例较大，说明组别之间的差异不显著，可能是由于随机误差的影响。因此，方差分析可以帮助我们确定不同因素对实验结果的影响程度，进一步揭示数据背后的规律和机制。方差可以分解成三个部分：*Q*=*Q*<sub>1</sub>+*Q*<sub>2</sub>+*Q*<sub>3</sub>。其中，*Q*<sub>1</sub>是指多个控制变量单独作用引起的平方和，可以用来描述每个变量单独是否存在影响；*Q*<sub>2</sub>是指多个控制变量交互作用引起的离差平方和，可以用来描述变量之间是否存在协同效应或交互；*Q*<sub>3</sub>则是随机扰动，用于反映结果受随机影响的程度。

在Python中，可以通过scipy.stats.f_oneway函数实现方差分析。

In [ ]:
from scipy.stats import f_oneway

# 假设有三组样本
group1 = np.random.normal(loc=0, scale=1, size=100)
group2 = np.random.normal(loc=0.5, scale=1, size=100)
group3 = np.random.normal(loc=1, scale=1, size=100)

# 进行ANOVA
f_stat, p_val = f_oneway(group1, group2, group3)

print(f'F-statistic: {f_stat}, p-value: {p_val}')

# 如果p值小于显著性水平（如0.05），则拒绝零假设（所有组的均值相等）
if p_val < 0.05:
    print("至少两组的均值存在显著差异")
else:
    print("所有组的均值没有显著差异")